In [ ]:
pip install Pillow

In [ ]:
from PIL import Image
import os
import glob

# CONFIGURACIÓN
CARPETA_ENTRADA = "logos"           # Carpeta con tus logos originales
CARPETA_SALIDA = "logos_optimizados" # Donde guardar los resultados
ANCHO_MAX = 240                     # px (ajusta según necesites)
CALIDAD_WEBP = 85                   # 0-100
CALIDAD_JPEG = 85

os.makedirs(CARPETA_SALIDA, exist_ok=True)

archivos = glob.glob(os.path.join(CARPETA_ENTRADA, "*"))
soportados = ('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff', '.webp')

total_original = 0
total_optimizado = 0

for ruta in archivos:
    ext = os.path.splitext(ruta)[1].lower()
    if ext not in soportados:
        continue

    nombre = os.path.basename(ruta)
    nombre_base = os.path.splitext(nombre)[0]

    try:
        img = Image.open(ruta)

        # Convertir a RGB si es necesario
        if img.mode in ('RGBA', 'P'):
            fondo = Image.new('RGB', img.size, (255, 255, 255))
            if img.mode == 'P':
                img = img.convert('RGBA')
            fondo.paste(img, mask=img.split()[3] if img.mode == 'RGBA' else None)
            img = fondo

        # Redimensionar manteniendo proporción
        if img.width > ANCHO_MAX:
            ratio = ANCHO_MAX / img.width
            nuevo_alto = int(img.height * ratio)
            img = img.resize((ANCHO_MAX, nuevo_alto), Image.LANCZOS)

        # Guardar como WebP (mejor compresión)
        ruta_webp = os.path.join(CARPETA_SALIDA, f"{nombre_base}.webp")
        img.save(ruta_webp, 'WEBP', quality=CALIDAD_WEBP, method=6)

        # Guardar también como JPEG de respaldo
        ruta_jpg = os.path.join(CARPETA_SALIDA, f"{nombre_base}.jpg")
        img.save(ruta_jpg, 'JPEG', quality=CALIDAD_JPEG, optimize=True)

        # Stats
        tam_original = os.path.getsize(ruta)
        tam_webp = os.path.getsize(ruta_webp)
        total_original += tam_original
        total_optimizado += tam_webp

        reduccion = (1 - tam_webp / tam_original) * 100
        print(f"✅ {nombre}: {tam_original/1024:.1f} KB → {tam_webp/1024:.1f} KB ({reduccion:.0f}% menos)")

    except Exception as e:
        print(f"❌ Error con {nombre}: {e}")

print(f"\n📊 TOTAL: {total_original/1024/1024:.2f} MB → {total_optimizado/1024/1024:.2f} MB")
ahorro = (1 - total_optimizado/total_original)*100 if total_original else 0
print(f"💾 Ahorro total: {ahorro:.1f}%")